# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Reading FlyRank's own paper ("The State of AI-Driven SEO in Numbers," March 2026) the same way
I'd want my own work read — constructively, specifically, and by asking where the label came
from before trusting the score.

### Finding A: "What Predicts Health?" (Random Forest feature importance, p.27)

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of
`health_score`, via a holdout-tested Random Forest.

**My methodology question:** *Where does the label come from, and do the features overlap with
its construction?* The paper's own methodology section defines `health_score` as
`impressions (30pts) + position (30pts) + CTR (20pts) + scroll_depth (20pts)` — a formula built
directly from four of the same signals fed into the model as features. The paper is admirably
upfront about this ("the target itself is partly constructed from some of these inputs, so
importance is descriptive rather than causal") — so my question isn't "did you know?", it's a
constructive follow-up: *would a second, fully independent-feature version of this analysis
(excluding position, impressions, CTR, and scroll depth entirely) reveal which non-constructive
signals actually relate to health — since the current chart can't distinguish "predicts health"
from "is arithmetically part of health"?*

### Finding B: "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)

The paper reports 71% holdout accuracy for a logistic regression separating growing from
declining pages, with an 80/20 split noted in the methodology section.

**My methodology question:** *Was this split grouped by brand, or random by row — and what does
71% look like next to the base rate?* Two things aren't stated: whether the 80/20 split holds
out entire brands (57 brands contribute rows here, and my own Week-5 work found a client-grouped
split can look meaningfully worse than a random one on the same data) or just shuffles rows; and
what the base rate for "growing" actually is. Finding #1 in this same paper reports 74.8K
growing vs. 45.6K declining pages — a 62.2% base rate for "up" — which would make 71% accuracy
only about 9 points of real lift, not the strong result it reads as in isolation.

Both questions follow the exact shape the `hunting-leakage-and-validating` skill trains: check
whether the label leaked into the features, and check whether the split and base rate were
disclosed next to the headline number.

In [1]:
# no execution needed here -- section 1 is a reading/methodology exercise against
# the PDF, not code against this repo's data. Section 2 below turns the same lens on my own model.
print("Paper: FlyRank -- The State of AI-Driven SEO in Numbers (March 2026)")
print("Findings audited: #Health-score RF feature importance (p.27), #Growth LogReg (p.29)")

Paper: FlyRank -- The State of AI-Driven SEO in Numbers (March 2026)
Findings audited: #Health-score RF feature importance (p.27), #Growth LogReg (p.29)


## 2. My model under an honest split (before/after)

My Week-5 notebook already used a client-grouped split from the start -- the right call, but it
means I never actually *saw* the gap a naive split would have hidden. Doing that comparison now,
in this notebook, on the same Random Forest and the same features as Week 5.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_features = ['search_volume', 'competition', 'word_count', 'content_age_days',
                     'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
                     'scroll_rate', 'ai_traffic_pct', 'impressions_90d', 'sessions_90d',
                     'clicks_90d']

def build_X(frame):
    X = frame[numeric_features].apply(pd.to_numeric, errors='coerce')
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

# --- BEFORE: naive random row split (what I did NOT do in Week 5, shown here for contrast) ---
X_all, y_all = build_X(df), df['is_declining_label']
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X_all, y_all, test_size=0.2,
                                               random_state=RANDOM_STATE, stratify=y_all)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                    random_state=RANDOM_STATE, n_jobs=-1)
rf_random.fit(Xtr_r, ytr_r)
scores_random = rf_random.predict_proba(Xte_r)[:, 1]

# --- AFTER: the same client-grouped split used in Week 5 ---
rng = np.random.default_rng(RANDOM_STATE)
clients = df['client_id'].dropna().unique()
shuffled = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = df['client_id'].isin(test_clients)
train_df, test_df = df[~test_mask].copy(), df[test_mask].copy()
Xtr_g, Xte_g = build_X(train_df), build_X(test_df)
ytr_g, yte_g = train_df['is_declining_label'], test_df['is_declining_label']
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                     random_state=RANDOM_STATE, n_jobs=-1)
rf_grouped.fit(Xtr_g, ytr_g)
scores_grouped = rf_grouped.predict_proba(Xte_g)[:, 1]

comparison = pd.DataFrame({
    'split': ['BEFORE -- random row split', 'AFTER -- client-grouped split'],
    'test_base_rate': [yte_r.mean(), yte_g.mean()],
    'precision@50': [precision_at_k(yte_r, scores_random, 50),
                      precision_at_k(yte_g, scores_grouped, 50)],
    'precision@20%': [precision_at_k(yte_r, scores_random, int(len(yte_r) * 0.2)),
                       precision_at_k(yte_g, scores_grouped, int(len(yte_g) * 0.2))],
})
print(comparison.round(3).to_string(index=False))
print(f"\nGap at precision@50: {comparison['precision@50'][0]:.3f} -> "
      f"{comparison['precision@50'][1]:.3f}  "
      f"(the random split overstates skill by {comparison['precision@50'][0]-comparison['precision@50'][1]:.3f})")
print("That gap IS the finding -- it's how much of the 'random split' score was the model")
print("partly memorizing which client a row came from, not genuine signal.")

                        split  test_base_rate  precision@50  precision@20%
   BEFORE -- random row split           0.542           0.9          0.809
AFTER -- client-grouped split           0.391           0.7          0.609

Gap at precision@50: 0.900 -> 0.700  (the random split overstates skill by 0.200)
That gap IS the finding -- it's how much of the 'random split' score was the model
partly memorizing which client a row came from, not genuine signal.


## 3. Leakage audit

Same attack-your-own-model checklist as Week 3, now run against the final Week-5 feature set
and model rather than a toy example.

- [x] Timeline drawn: all 13 features are pre-decision observable signals (position, CTR,
      impressions, word count, staleness, etc.) — none depend on the outcome window.
- [x] No label-derived or sibling columns in the features — `trend_direction`/`trend_pct` (what
      the label is built from) were never included. Proven below by deliberately adding one back.
- [x] No product flags as features — none exist in this dataset to begin with (per the
      `flyrank-data` skill), so nothing to strip.
- [x] Split grouped by the repeating entity (client) — shown in section 2 above.
- [x] Base rate printed next to every metric — done throughout.
- [x] Top feature importance sanity-checked — `impressions_90d` and `avg_position` lead, which
      matches the lane's own logic rather than looking suspiciously perfect.
- [x] Metrics recomputed out-of-fold (held-out test clients), never in-sample.

In [3]:
# Deliberately add trend_pct back in -- the exact column the label is derived from --
# and watch the score jump toward perfect. Then remove it and keep the honest number.
X_leak_tr = Xtr_g.copy(); X_leak_tr['trend_pct_LEAK'] = train_df['trend_pct'].values
X_leak_te = Xte_g.copy(); X_leak_te['trend_pct_LEAK'] = test_df['trend_pct'].values

rf_leak = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                  random_state=RANDOM_STATE, n_jobs=-1)
rf_leak.fit(X_leak_tr, ytr_g)
scores_leak = rf_leak.predict_proba(X_leak_te)[:, 1]

auc_honest = roc_auc_score(yte_g, scores_grouped)
auc_leak = roc_auc_score(yte_g, scores_leak)
p50_honest = precision_at_k(yte_g, scores_grouped, 50)
p50_leak = precision_at_k(yte_g, scores_leak, 50)

print(f"HONEST  (no trend_pct):  AUC={auc_honest:.3f}   precision@50={p50_honest:.3f}")
print(f"LEAKY   (+ trend_pct):   AUC={auc_leak:.3f}   precision@50={p50_leak:.3f}")
print(f"\nJump: AUC {auc_honest:.3f} -> {auc_leak:.3f}  (a jump to a perfect 1.000 is the confession)")
print("Deleting the leaked column now -- keeping only the honest 0.741 / 0.700 as the real result.")
del X_leak_tr, X_leak_te, rf_leak, scores_leak

HONEST  (no trend_pct):  AUC=0.741   precision@50=0.700
LEAKY   (+ trend_pct):   AUC=1.000   precision@50=1.000

Jump: AUC 0.741 -> 1.000  (a jump to a perfect 1.000 is the confession)
Deleting the leaked column now -- keeping only the honest 0.741 / 0.700 as the real result.


## 4. Claim rewrite

Auditing my own Week-5 notebook (`w05_model.ipynb`) the same way I read the paper above.

**Original claim (Week 5, section 4):**
> "Random Forest wins clearly on both cuts (precision@50 ≈ 0.70, precision@20% ≈ 0.61, vs. the
> rule's 0.46 / 0.56)."

**What's actually true, now that I've seen the before/after gap:** the 0.70/0.61 numbers are
from the honest client-grouped split — that part holds up. But that split only holds out **6
clients**. "Clearly" implies a level of confidence that a 6-group holdout can't really support;
a different random seed choosing a different 6 clients could plausibly move these numbers by a
meaningful amount, and I have no measure of that variance in Week 5.

**Rewritten, public-safe version:**
> "On this client-grouped split, Random Forest outperformed both the rule baseline and Logistic
> Regression at precision@50 and precision@20% (0.70/0.61 vs. the rule's 0.46/0.56). With only 6
> held-out clients backing this comparison, this result is directional rather than conclusive —
> a different client split could shift these numbers, and that variance hasn't been measured
> here."

That rewrite trades a confident-sounding sentence for an honest one — exactly the move the
paper itself makes when it says RF's health-score importance is "descriptive rather than
causal" instead of claiming discovery.

In [4]:
# nothing further to run -- section 4 is a text audit of Week 5's own markdown,
# cross-checked against the split-size evidence computed in section 2 above.
print("Claim rewritten. See markdown above.")

Claim rewritten. See markdown above.
